# Día 4 · Unidad 4 — Funciones (IV): lambda, map y filter

**Objetivos de aprendizaje**
- Escribir funciones anónimas de una sola expresión con `lambda`.
- Aplicar una función a cada elemento de un iterable con `map()`.
- Seleccionar elementos de un iterable según una condición con `filter()`.
- Combinar `lambda` con `map()`/`filter()` en casos reales de valoración de riesgo.

**Duración estimada:** 45 minutos.

**Requisitos previos:** `01_teoria_funciones_basicas.ipynb` de este mismo día.

## 1. Expresiones lambda

Una expresión `lambda` crea una función "anónima" (sin nombre) en una sola línea, sin usar `def`. Su cuerpo es **una única expresión**, no un bloque de sentencias — esa es la diferencia clave con una función normal.

In [1]:
def elevar_al_cuadrado(num):
    resultado = num ** 2
    return resultado


elevar_al_cuadrado(4)


16

Simplificando, primero quitando la variable intermedia...

In [2]:
def elevar_al_cuadrado(num):
    return num ** 2


elevar_al_cuadrado(4)


16

...y ahora como `lambda`, que es exactamente la misma lógica sin nombre ni `def`:

In [3]:
lambda num: num ** 2


<function __main__.<lambda>(num)>

Para poder llamarla, hace falta asignarla a un nombre — aunque en la práctica casi nunca se hace así (si le pones nombre, mejor usa `def`, es más legible; el valor real de `lambda` está en pasarla "al vuelo" como argumento de otra función, como verás en las secciones 2 y 3):

In [4]:
elevar_al_cuadrado_lambda = lambda num: num ** 2
elevar_al_cuadrado_lambda(4)


16

## 2. La función `map()`

`map(funcion, iterable)` aplica `funcion` a cada elemento de `iterable`.

In [5]:
def prima_con_recargo(prima: float) -> float:
    """Aplica un recargo fijo del 8% a una prima.

    Args:
        prima: prima base.

    Returns:
        float: prima con el recargo aplicado.
    """
    return prima * 1.08


primas = [420.0, 310.5, 275.0, 590.25]
primas


[420.0, 310.5, 275.0, 590.25]

> **Importante:** al pasar una función como argumento de `map()`, va **sin** paréntesis — le pasas la función en sí, no el resultado de llamarla.

In [6]:
map(prima_con_recargo, primas)


`map()` devuelve un *generator*: un objeto que calcula los resultados uno a uno, sólo cuando se le piden. Para verlos todos de golpe, lo convertimos a lista:

In [7]:
list(map(prima_con_recargo, primas))


[453.6, 335.34000000000003, 297.0, 637.47]

Con `lambda`, no hace falta ni siquiera definir la función aparte si sólo se usa una vez:

In [8]:
list(map(lambda prima: prima * 1.08, primas))


[453.6, 335.34000000000003, 297.0, 637.47]

## 3. La función `filter()`

`filter(funcion, iterable)` se queda sólo con los elementos de `iterable` para los que `funcion` devuelve `True`.

In [9]:
def prima_alta(prima: float) -> bool:
    """Indica si una prima supera el umbral de revisión manual (500).

    Args:
        prima: prima a evaluar.

    Returns:
        bool: True si la prima supera 500.
    """
    return prima > 500


In [10]:
filter(prima_alta, primas)


Igual que `map()`, hay que convertirlo a lista (u otro iterable) para ver el resultado:

In [11]:
list(filter(prima_alta, primas))


[590.25]

In [12]:
list(filter(lambda prima: prima > 500, primas))


[590.25]

## 4. Combinando `lambda` con `map()` y `filter()`: política de suscripción

Un caso más realista: dada una lista de combinaciones (frecuencia, severidad), calculamos la prima de riesgo de cada una y aplicamos una política de suscripción simple: si la prima de riesgo es demasiado alta, el riesgo se rechaza.

In [13]:
from typing import Union

def politica_suscripcion(combinacion: tuple[float, float]) -> Union[float, str]:
    """Aplica una política de suscripción sencilla según frecuencia y severidad.

    Args:
        combinacion: tupla (frecuencia, severidad).

    Returns:
        Union[float, str]: la prima de riesgo (frecuencia * severidad) si es
        menor que 50; en caso contrario, el texto "Riesgo rechazado".
    """
    frecuencia, severidad = combinacion
    prima_riesgo = frecuencia * severidad
    if prima_riesgo >= 50:
        return "Riesgo rechazado"
    return prima_riesgo


frec_sev = [(0.1, 100), (0.5, 200), (0.05, 400), (0.02, 350)]
list(map(politica_suscripcion, frec_sev))


[10.0, 'Riesgo rechazado', 20.0, 7.0]

Ahora, combinado con `filter()` y una `lambda`, nos quedamos sólo con las combinaciones que **superan** el umbral de rechazo, antes incluso de aplicarles la política:

In [14]:
list(filter(lambda combinacion: combinacion[0] * combinacion[1] >= 50, frec_sev))


[(0.5, 200)]

> 📊 **En Excel esto sería...** `map()` es como arrastrar una fórmula hacia abajo por una columna entera (aplicar la misma operación a cada fila); `filter()` es como un Autofiltro con una condición (quedarte sólo con las filas que la cumplen).

**Para ti:** dada `matriculas = ["1911LSQ", "1234ABS", "1145LMS"]`, usa `map()` con una `lambda` para quedarte sólo con los 4 caracteres numéricos de cada matrícula, como una lista de enteros (`[1911, 1234, 1145]`).

**Para ti:** dada la lista `primas` de este notebook, usa `filter()` con una `lambda` para quedarte con las primas que están **entre** 300 y 500 (ambos inclusive). Después, aplícales `map()` con `prima_con_recargo` para ver el resultado tras el recargo — encadena ambas llamadas en una sola expresión.

## Resumen

- `lambda parametros: expresion` crea una función anónima de una única expresión, sin `def` ni `return` explícito.
- `lambda` gana su sentido cuando se pasa "al vuelo" como argumento de otra función (`map`, `filter`...); si le pones nombre para reutilizarla, mejor usa `def`.
- `map(funcion, iterable)` aplica `funcion` a cada elemento y devuelve un *generator*; conviértelo con `list(...)` para verlo.
- `filter(funcion, iterable)` se queda sólo con los elementos para los que `funcion` devuelve un valor verdadero.
- Al pasar una función como argumento de `map()`/`filter()`, va sin paréntesis: le pasas la función, no el resultado de llamarla.
- `map()` y `filter()` se pueden encadenar para construir pipelines de transformación y selección en una línea, muy legibles una vez te acostumbras a leerlos.

**Siguiente:** `05_teoria_decoradores_bonus.ipynb` y `06_teoria_functools_bonus.ipynb` (contenido opcional/bonus), o directamente `07_ejercicios.ipynb` si el grupo va justo de tiempo.